# Building an LLM from scratch: Mini-GPT with PyTorch
**Code.Hub · Hands-on module**

In this notebook we build, step by step, a complete **decoder-only language model** (the same architecture as GPT, Llama, Claude and others), train it and make it write text.

**We open the "black box" of LLMs, piece by piece.** We understand every part of an LLM by writing it: what a token is, why there is a context window, what `temperature` does, why APIs charge per token.

| Session | Topic | What we have at the end |
|---|---|---|
| **1** | What an LLM is · PyTorch · Tokenization & embeddings | An "LLM" built from simple counting, and our text turned into numbers |
| **2** | Positional encodings · Self-attention · Causal attention | A working attention head (`Head`) |
| **3** | Multi-head attention · Decoder block · Full model | `MiniGPT`, ready but untrained |
| **4** | Training · Text generation · Today's landscape | A trained model that writes "Shakespearean" text |

### How to run it
1. Open the notebook in **Google Colab** (File → Upload notebook).
2. Optional but recommended: **Runtime → Change runtime type → T4 GPU**. It also works on CPU; training is just a bit slower.
3. Run the cells **in order** (Shift + Enter).
4. If you start in the middle (e.g. Session 3 with a fresh runtime), first run the **⏩ CATCH-UP** cell at the start of that session.

> **Shape convention** we will see everywhere: `B` = batch (how many pieces of text at once), `T` = time (how many tokens in each piece), `C` = channels (the size of each token's vector).

---
# Session 1 · What an LLM is, PyTorch, Tokenization

## 1.1 What an LLM is: next-token prediction

A Large Language Model does **one thing only**: you give it some text and it returns **a probability distribution over the next token**.

| Input | Possible next tokens |
|---|---|
| `The capital of Greece is` | `Athens` 0.93 · `the` 0.03 · `a` 0.01 · … |

Mathematically: $P(\text{token}_{t+1} \mid \text{token}_1, \dots, \text{token}_t)$

**So how does it write whole answers?** By repetition (autoregressive generation):
1. it predicts the next token,
2. appends it to the end of the text,
3. asks the model again with the longer text, over and over.

**How is it trained? (language-modelling objective)** We give it huge amounts of text and "penalise" it whenever it assigns a low probability to the token that *actually* comes next. Nobody needs to add labels: **the labels are the text itself, shifted by one position** (self-supervised learning).

Let's see it in practice with the simplest "language model" there is: **counting**.

### Setup and dataset
We will use *Tiny Shakespeare*: about 1 million characters from Shakespeare's plays. It is small enough to train quickly, and "recognisable" enough to tell whether the model is learning.

In [ ]:
import math, os, time, random
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

In [ ]:
import urllib.request

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(URL, "input.txt")

with open("input.txt", encoding="utf-8") as f:
    text = f.read()
print("Characters in the dataset:", len(text))

In [ ]:
print(text[:400])

### Our first "language model": counting which character follows which
For every character, we count how many times each other character follows it. This is already a (very dumb) language model: it knows $P(\text{next} \mid \text{current})$.

In [ ]:
from collections import Counter, defaultdict

next_counts = defaultdict(Counter)
for a, b in zip(text, text[1:]):
    next_counts[a][b] += 1

print("After 'q':", next_counts["q"].most_common(3))
print("After 't':", next_counts["t"].most_common(5))

Now let's make it **write**, picking the most likely next character every time (this is called **greedy** decoding):

In [ ]:
def predict_next(ch):
    return next_counts[ch].most_common(1)[0][0]

s = "T"
for _ in range(40):
    s += predict_next(s[-1])
print(s)

It gets stuck in a loop (`the the the…`). Let's try **picking at random, according to the probabilities** (sampling):

In [ ]:
def sample_next(ch):
    options = next_counts[ch]
    return random.choices(list(options.keys()), weights=list(options.values()))[0]

s = "T"
for _ in range(200):
    s += sample_next(s[-1])
print(s)

### What we learned from this experiment
- **Generation = predict the next token + repeat.** This is what ChatGPT and Claude do too.
- **Greedy vs sampling:** greedy is predictable but gets stuck; sampling has variety but also "madness". In Session 4 we will see the tools that balance the two (`temperature`, `top_k`, `top_p`).
- **The big problem:** our model sees **only the last character**. It has no idea what was said before.

> **The idea of the whole course:** we will replace this "table of counts" with a **neural network that sees all the previous text** (the context). That network is the **Transformer**.

## 1.2 PyTorch essentials

PyTorch is the library most LLMs are built with. We only need 4 concepts:

| Concept | What it is, in one sentence |
|---|---|
| **Tensor** | A multi-dimensional array of numbers (like a NumPy array) that can also run on a GPU. |
| **Autograd** | PyTorch computes derivatives (gradients) automatically. |
| **`nn.Module`** | The base class for every piece of a neural network. |
| **Training loop** | 4 steps repeated over and over: forward → loss → backward → step. |

### Tensors
The most important thing about a tensor is its **shape**. Half of all bugs in deep learning are wrong shapes.

In [ ]:
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
print("shape:", x.shape, "| dtype:", x.dtype)
print(x * 10)            # element-wise operation
print(x.sum(dim=1))      # sum per row
print(x @ x.T)           # matrix multiplication: (2,3) @ (3,2) -> (2,2)

In [ ]:
# In LLMs we constantly see 3-dimensional tensors: (B, T, C)
batch = torch.randn(4, 8, 16)   # 4 texts, 8 tokens each, a vector of 16 numbers per token
print(batch.shape)
print(batch[0, 0])              # the vector of the 1st token of the 1st text

### Autograd
If a tensor has `requires_grad=True`, PyTorch remembers every operation we perform on it. With `.backward()` it computes how much the result changes if we change the tensor a little: the **derivative** (gradient).

Example: $y = w^2 + 2w$, so $\frac{dy}{dw} = 2w + 2$. For $w=3$ we expect **8**.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
y = w**2 + 2*w
y.backward()
print("dy/dw =", w.grad)

> **Why do we care?** The gradient tells us **in which direction to change the parameters to reduce the error**. An LLM has millions or billions of such parameters, and autograd finds the gradient for all of them at once.

### `nn.Module` and the training loop
We will "train" the smallest network in the world: to learn the relationship $y = 2x + 1$ from noisy data.

In [ ]:
X = torch.linspace(-1, 1, 100).unsqueeze(1)          # (100, 1)
Y = 2 * X + 1 + 0.1 * torch.randn_like(X)           # (100, 1) with a little noise

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)                # y = w*x + b, with w and b being learned

    def forward(self, x):
        return self.linear(x)

tiny = TinyModel()
print(list(tiny.parameters()))

In [ ]:
optimizer = torch.optim.SGD(tiny.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

for step in range(200):
    pred = tiny(X)                  # 1. forward: make a prediction
    loss = loss_fn(pred, Y)         # 2. loss: how wrong is it?
    optimizer.zero_grad()           #    (clear the old gradients)
    loss.backward()                 # 3. backward: compute gradients
    optimizer.step()                # 4. step: nudge the parameters
    if step % 40 == 0:
        print(f"step {step:3d} | loss {loss.item():.4f}")

print("w =", round(tiny.linear.weight.item(), 3), "| b =", round(tiny.linear.bias.item(), 3))

> **Remember these 4 steps.** In Session 4 we will train our GPT with **exactly the same loop**. Only the model changes (a Transformer instead of `nn.Linear(1,1)`) and the loss (cross-entropy instead of MSE).

## 1.3 Tokenization, vocabulary and token embeddings

Neural networks don't read letters, they read **numbers**. **Tokenization** splits text into pieces (tokens) and gives each one an **ID**. The set of all possible tokens is the **vocabulary**.

| Method | Example for `"unhappiness"` | Vocabulary size | Sequence length |
|---|---|---|---|
| Per character | `u n h a p p i n e s s` | very small (~100) | very long |
| Per word | `unhappiness` | huge, plus unknown words | short |
| **Subword (BPE)** | `un happi ness` | ~50k–200k | balanced, **this is what modern LLMs use** |

For simplicity, we will do **per-character tokenization**. The logic is exactly the same.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # string -> int
itos = {i: ch for ch, i in stoi.items()}       # int -> string

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

print("vocab_size:", vocab_size)

In [ ]:
print("".join(chars))
print(encode("hello world"))
print(decode(encode("hello world")))

### (Optional) How a real model does tokenization
`tiktoken` is the BPE tokenizer used by OpenAI models. Notice how many **more** tokens Greek "costs": that's why a Greek prompt is often more expensive in an API.

In [ ]:
!pip install -q tiktoken
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
for s in ["Hello, how are you today?", "Γεια σου, πώς είσαι σήμερα;"]:
    ids = enc.encode(s)
    print(f"{len(s):3d} characters -> {len(ids):3d} tokens | {s}")
print([enc.decode([i]) for i in enc.encode("unhappiness is temporary")])

### Token embeddings
An ID (e.g. `46`) is just a numeric label; it says nothing about meaning. That's why each token is mapped to a **vector** (embedding) of `C` numbers.

`nn.Embedding(vocab_size, C)` is simply a **lookup table** with `vocab_size` rows: the token with ID 46 gets row 46. At first the numbers are random; during training they are **learned**, so that tokens with similar "behaviour" end up with similar vectors.

In [ ]:
emb = nn.Embedding(vocab_size, 8)          # vocabulary of 65 tokens, a vector of 8 numbers each
print("Embedding table:", emb.weight.shape)

ids = torch.tensor(encode("hi"))
vectors = emb(ids)
print("ids:", ids.tolist(), "-> shape", vectors.shape)
print(vectors)

### From text to training examples
We encode the whole text and split it into **train (90%)** and **validation (10%)**. The model never sees the validation part during training: we use it to check whether it **generalises** or just memorises.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(data.shape, train_data.shape, val_data.shape)

And now the most important cell of the session. A chunk of 8 tokens is not **one** training example, it is **eight**: at each position, the target is simply the next character.

In [ ]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]      # the same text, shifted by 1

for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"context {decode(context.tolist())!r:12} -> target {decode([target.item()])!r}")

### Recap Session 1
- An LLM **predicts the next token**; it writes text by repeating that prediction.
- It is trained on text where **the target is the text shifted by 1**.
- PyTorch = tensors + autograd + `nn.Module` + the 4-step loop.
- Text → **tokens** → **IDs** → **embeddings** (vectors that are learned).

**Next session:** how tokens "talk" to each other, i.e. **attention**.

---
# Session 2 · Positional encodings, Self-attention, Causal attention

In [ ]:
# ⏩ CATCH-UP: run this ONLY if you are starting with a fresh runtime
# (rebuilds everything we need from the previous sessions)

import math, os, time, random
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)
import urllib.request

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(URL, "input.txt")

with open("input.txt", encoding="utf-8") as f:
    text = f.read()
print("Characters in the dataset:", len(text))
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # string -> int
itos = {i: ch for ch, i in stoi.items()}       # int -> string

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

print("vocab_size:", vocab_size)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(data.shape, train_data.shape, val_data.shape)
print("✅ Ready")

## 2.1 Positional encodings

Attention (which we will see shortly) has a strange property: **it doesn't know the order of the tokens**. So "the dog bit the man" and "the man bit the dog" would look the same.

**Solution:** to each token embedding we **add** a second vector that encodes its **position**.

$$x = \text{token\_embedding}(\text{id}) + \text{position\_embedding}(\text{position})$$

There are two classic approaches:
- **Learned:** one more `nn.Embedding(block_size, C)`, one row per position. This is what GPT-2 does, and what we will do.
- **Sinusoidal:** fixed sine/cosine waves, from the original Transformer paper (2017).

Modern models often use more advanced variants (e.g. RoPE), but the idea is the same: **every token should know where it is**.

In [ ]:
torch.manual_seed(0)
n_embd = 16
block_size = 8

tok_emb = nn.Embedding(vocab_size, n_embd)
pos_emb = nn.Embedding(block_size, n_embd)

idx = torch.tensor([encode("to be or")])     # (B=1, T=8)
B, T = idx.shape
tok = tok_emb(idx)                           # (1, 8, 16)
pos = pos_emb(torch.arange(T))               # (8, 16)
x = tok + pos                                # broadcasting -> (1, 8, 16)
print(tok.shape, pos.shape, x.shape)

The letter `o` appears at positions 1 and 6. Its **token embedding** is the same at both; after adding the position embedding, **it no longer is**:

In [ ]:
print("same token embedding?", torch.allclose(tok[0, 1], tok[0, 6]))
print("same final x?        ", torch.allclose(x[0, 1], x[0, 6]))

### (Optional) What sinusoidal encodings look like
Each row is a position, each column a dimension. Every position has its own unique "fingerprint".

In [ ]:
import matplotlib.pyplot as plt

def sinusoidal(T, C):
    pos = torch.arange(T).unsqueeze(1)
    div = torch.exp(torch.arange(0, C, 2) * (-math.log(10000.0) / C))
    pe = torch.zeros(T, C)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

plt.figure(figsize=(8, 4))
plt.imshow(sinusoidal(64, 32), cmap="RdBu", aspect="auto")
plt.xlabel("dimension"); plt.ylabel("position"); plt.colorbar(); plt.show()

## 2.2 Self-attention: queries, keys, values

So far every token has its own vector, but **knows nothing about the others**. Self-attention is the mechanism by which each token **gathers information from the other tokens**.

**A library-search analogy.** Each token produces three vectors:

| | Question it answers | Analogy |
|---|---|---|
| **Query (Q)** | "What am I looking for?" | What you type into the search engine |
| **Key (K)** | "What do I contain?" | The label on the spine of each book |
| **Value (V)** | "What do I give if you pick me?" | The content of the book |

The steps:
1. **Scores:** we compare each token's query with everyone's keys (dot product, `q @ k.T`). High score = "relevant".
2. **Scale:** we divide by $\sqrt{d_k}$ (the key size), so the scores don't get too large.
3. **Softmax:** we turn the scores into **weights** that sum to 1.
4. **Weighted sum:** each token gets a weighted mix of the values.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Q, K and V come from three `nn.Linear` layers applied to the same `x`, and their weights are **learned**: the model learns by itself what to look for.

In [ ]:
torch.manual_seed(0)
head_size = 8
query = nn.Linear(n_embd, head_size, bias=False)
key   = nn.Linear(n_embd, head_size, bias=False)
value = nn.Linear(n_embd, head_size, bias=False)

q = query(x)    # (1, 8, 8)
k = key(x)      # (1, 8, 8)
v = value(x)    # (1, 8, 8)
print(q.shape, k.shape, v.shape)

In [ ]:
scores = q @ k.transpose(-2, -1)          # (1, 8, 8): how well each token "matches" every other
scores = scores / head_size ** 0.5        # scale
weights = F.softmax(scores, dim=-1)       # each row sums to 1
out = weights @ v                         # (1, 8, 8)

print(weights[0].round(decimals=2))
print("sum of each row:", weights[0].sum(dim=-1))
print("out:", out.shape)

**Row i** of the `weights` matrix says: "to build its new representation, token i takes this much from token 0, this much from token 1…".

### Why do we divide by $\sqrt{d_k}$?
Dot products grow as the dimension grows. Very large scores make the softmax almost **one-hot** (all the weight on one token), and then the model struggles to learn.

In [ ]:
a, b = torch.randn(1000, 64), torch.randn(1000, 64)
raw = (a * b).sum(dim=-1)
print("variance without scale:", round(raw.var().item(), 1), "| with scale:", round((raw / 64 ** 0.5).var().item(), 2))

s = torch.tensor([1.0, 2.0, 3.0])
print("softmax(s)     :", F.softmax(s, dim=-1).round(decimals=3))
print("softmax(s * 10):", F.softmax(s * 10, dim=-1).round(decimals=3), "<- almost one-hot")

## 2.3 Causal (masked) self-attention: why decoders are autoregressive

There is a problem with the above: token 3 also looks at tokens 4, 5, 6, 7, i.e. **the future**. During training this is **cheating**: if we want it to predict the next token, it must not already see it.

**Solution: the mask.** Before the softmax, we put $-\infty$ in every position "above the diagonal". The softmax of $-\infty$ is 0, so **every token sees only itself and the ones before it**.

This gives us two things:
- **In training:** we learn **all positions in parallel** in one forward pass (remember: 8 examples from one 8-token chunk), without any position seeing its answer.
- **In generation:** the model is **autoregressive**: it produces one token at a time and feeds it back to itself.

> **Decoder vs encoder:** BERT-style models (encoders) look in all directions and are good at *understanding* (classification, embeddings). GPT, Llama and Claude are **decoder-only**: causal attention, built for *generating* text.

In [ ]:
T = 8
mask = torch.tril(torch.ones(T, T))
print(mask)

In [ ]:
scores = q @ k.transpose(-2, -1) / head_size ** 0.5
scores = scores.masked_fill(mask == 0, float("-inf"))
weights = F.softmax(scores, dim=-1)
print(weights[0].round(decimals=2))

A lower-triangular matrix: row 0 gives all its weight to itself, row 7 spreads weight over all 8 positions.

### Putting it all in one class: `Head`
We will reuse this class in Session 3. `register_buffer` stores the mask inside the module without it being a trainable parameter.

In [ ]:
class Head(nn.Module):
    """One head of causal self-attention."""
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)      # (B, T, hs)
        q = self.query(x)    # (B, T, hs)
        v = self.value(x)    # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5       # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v       # (B, T, hs)

In [ ]:
head = Head(n_embd=16, head_size=8, block_size=8)
out = head(x)
print(x.shape, "->", out.shape)

### Proof that it doesn't "cheat"
We change **only the last** token. If the mask works, the outputs of all previous positions **must stay the same**.

In [ ]:
x2 = x.clone()
x2[0, -1] = torch.randn(n_embd)          # change the last token

out1, out2 = head(x), head(x2)
print("previous positions unchanged?", torch.allclose(out1[0, :-1], out2[0, :-1]))
print("last position unchanged?     ", torch.allclose(out1[0, -1], out2[0, -1]))

### Recap Session 2
- **Positional embeddings:** we add the position to the token embedding, because attention doesn't know the order.
- **Self-attention:** each token asks (Q), is compared with the keys (K) and gathers values (V): $\text{softmax}(QK^T/\sqrt{d_k})V$.
- **Causal mask:** no token sees the future, so we learn all positions in parallel and generate text one token at a time.

**Next session:** many heads together, the decoder block and the whole model.

---
# Session 3 · Multi-head attention, Decoder block, Full model

In [ ]:
# ⏩ CATCH-UP: run this ONLY if you are starting with a fresh runtime
# (rebuilds everything we need from the previous sessions)

import math, os, time, random
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)
import urllib.request

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(URL, "input.txt")

with open("input.txt", encoding="utf-8") as f:
    text = f.read()
print("Characters in the dataset:", len(text))
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # string -> int
itos = {i: ch for ch, i in stoi.items()}       # int -> string

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

print("vocab_size:", vocab_size)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(data.shape, train_data.shape, val_data.shape)
class Head(nn.Module):
    """One head of causal self-attention."""
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)      # (B, T, hs)
        q = self.query(x)    # (B, T, hs)
        v = self.value(x)    # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5       # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v       # (B, T, hs)
print("✅ Ready")

## 3.1 Multi-head attention

One attention head can learn **one** kind of relationship. But language has many at once: what the previous letter is, whether we are inside a word, who is speaking, which verb goes with which subject…

**Solution:** we run **several smaller heads in parallel**, each with its own Q, K, V, and concatenate the results.

- If `n_embd = 64` and `n_head = 4`, each head has `head_size = 64 / 4 = 16`.
- 4 heads × 16 = 64: the output has **the same size as the input**.
- At the end, a linear projection (`proj`) "mixes" what the heads found.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Several heads in parallel, concatenated and projected."""
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)]
        )
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (B, T, n_embd)
        return self.dropout(self.proj(out))

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 8, 64)                                   # (B=2, T=8, C=64)
mha = MultiHeadAttention(n_embd=64, n_head=4, block_size=8)
print(x.shape, "->", mha(x).shape)
print("heads:", len(mha.heads), "| head_size:", mha.heads[0].key.out_features)

## 3.2 The decoder block: feed-forward, residual connections, layer norm

A Transformer **block** has two phases:

| Phase | What it does | Analogy |
|---|---|---|
| **Multi-head attention** | Tokens **communicate** with each other | A team meeting: you exchange information |
| **Feed-forward (MLP)** | Each token **processes on its own** what it gathered | After the meeting, everyone thinks at their own desk |

And two "technical" pieces that make deep networks trainable:

**Residual connections:** instead of `x = f(x)` we write `x = x + f(x)`. Each block **adds a correction** to the information instead of replacing it. This way information (and gradients) flow freely through dozens of blocks, like on a "highway".

**Layer normalization:** normalises each token's vector to mean ≈ 0 and standard deviation ≈ 1. It keeps the numbers within sensible bounds and stabilises training. We place it **before** each phase (*pre-norm*), like most modern models.

```
x ──► LayerNorm ──► Multi-Head Attention ──►(+)──► LayerNorm ──► FeedForward ──►(+)──► output
│                                            ▲  │                                ▲
└────────────────── residual ────────────────┘  └──────────── residual ──────────┘
```

In [ ]:
class FeedForward(nn.Module):
    """A small MLP applied to each token independently."""
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
ln = nn.LayerNorm(64)
y = ln(x)
print("before: mean", round(x[0, 0].mean().item(), 3), "| std", round(x[0, 0].std().item(), 3))
print("after : mean", round(y[0, 0].mean().item(), 3), "| std", round(y[0, 0].std().item(), 3))

In [ ]:
class Block(nn.Module):
    """Decoder block: communication (attention) + computation (feed-forward)."""
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # residual connection #1
        x = x + self.ffwd(self.ln2(x))   # residual connection #2
        return x

In [ ]:
block = Block(n_embd=64, n_head=4, block_size=8)
print(x.shape, "->", block(x).shape)

> **Notice:** a block takes `(B, T, C)` and returns `(B, T, C)`. That is exactly why we can stack as many as we want on top of each other.

## 3.3 Stacking blocks into a full decoder-only model

```
 token IDs (B, T)
      │
      ▼
 Token embedding + Position embedding      → (B, T, C)
      │
      ▼
 Block × N   (attention + feed-forward)    → (B, T, C)
      │
      ▼
 Final LayerNorm                           → (B, T, C)
      │
      ▼
 Linear "lm_head"                          → (B, T, vocab_size)   ← logits
```

The output is called **logits**: for **every position**, a score for **every token in the vocabulary**. If we pass the logits through softmax, we get the probability distribution for the next token. That is where Session 1 started.

If we also pass `targets`, the model also computes the **loss** (cross-entropy). We will explain it in detail in Session 4.

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=64, n_head=4, n_layer=4, block_size=32, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)       # what token it is
        self.pos_emb = nn.Embedding(block_size, n_embd)       # which position it is at
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)          # -> one score per vocabulary token

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)                                    # (B, T, C)
        pos = self.pos_emb(torch.arange(T, device=idx.device))     # (T, C)
        x = tok + pos                                              # (B, T, C)
        x = self.blocks(x)                                         # (B, T, C)
        x = self.ln_f(x)                                           # (B, T, C)
        logits = self.lm_head(x)                                   # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

In [ ]:
model = MiniGPT(vocab_size, n_embd=64, n_head=4, n_layer=4, block_size=32).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

Let's give it a prompt and see what shapes come out:

In [ ]:
idx = torch.tensor([encode("ROMEO:")], device=device)   # (1, 6)
logits, _ = model(idx)
print("logits:", logits.shape)                          # (1, 6, 65)

probs = F.softmax(logits[0, -1], dim=-1)               # distribution for the next token
top = torch.topk(probs, 5)
print([(decode([i]), round(p, 3)) for p, i in zip(top.values.tolist(), top.indices.tolist())])

### Sanity check: the loss of an untrained model
A model that knows nothing gives roughly **the same probability to all** 65 tokens: $1/65$. The cross-entropy is then $-\ln(1/65) = \ln(65) \approx 4.17$. If we see something close to that, our model is wired correctly.

In [ ]:
xb = train_data[:32].unsqueeze(0).to(device)       # (1, 32)
yb = train_data[1:33].unsqueeze(0).to(device)      # (1, 32), shifted by 1
_, loss = model(xb, yb)
print("loss:", round(loss.item(), 3), "| expected ≈", round(math.log(vocab_size), 3))

### Generation with an untrained model
We write the simplest generation loop (greedy): predict → take the most likely → append it → again. Notice `idx[:, -model.block_size:]`: the model sees **at most `block_size` tokens**. This is the **context window**.

In [ ]:
@torch.no_grad()
def generate_greedy(model, idx, max_new_tokens):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]               # crop to the context window
        logits, _ = model(idx_cond)                         # (B, T, vocab)
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)   # (B, 1)
        idx = torch.cat([idx, next_id], dim=1)              # append and repeat
    return idx

out = generate_greedy(model, idx, max_new_tokens=100)
print(decode(out[0].tolist()))

Garbage, as expected: the model exists, but **hasn't learned anything yet**.

### Recap Session 3
- **Multi-head attention:** many small heads in parallel, each learning a different relationship.
- **Block** = attention (communication) + feed-forward (computation), with **residuals** and **LayerNorm**.
- **MiniGPT** = embeddings → N blocks → LayerNorm → `lm_head` → logits.
- Untrained loss ≈ ln(65) ≈ 4.17. The **context window** is `block_size`.

**Next session:** we train it and make it write.

---
# Session 4 · Training, Text generation, Today's landscape

In [ ]:
# ⏩ CATCH-UP: run this ONLY if you are starting with a fresh runtime
# (rebuilds everything we need from the previous sessions)

import math, os, time, random
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)
import urllib.request

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(URL, "input.txt")

with open("input.txt", encoding="utf-8") as f:
    text = f.read()
print("Characters in the dataset:", len(text))
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # string -> int
itos = {i: ch for ch, i in stoi.items()}       # int -> string

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

print("vocab_size:", vocab_size)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(data.shape, train_data.shape, val_data.shape)
class Head(nn.Module):
    """One head of causal self-attention."""
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)      # (B, T, hs)
        q = self.query(x)    # (B, T, hs)
        v = self.value(x)    # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5       # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v       # (B, T, hs)
class MultiHeadAttention(nn.Module):
    """Several heads in parallel, concatenated and projected."""
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)]
        )
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (B, T, n_embd)
        return self.dropout(self.proj(out))
class FeedForward(nn.Module):
    """A small MLP applied to each token independently."""
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)
class Block(nn.Module):
    """Decoder block: communication (attention) + computation (feed-forward)."""
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # residual connection #1
        x = x + self.ffwd(self.ln2(x))   # residual connection #2
        return x
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=64, n_head=4, n_layer=4, block_size=32, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)       # what token it is
        self.pos_emb = nn.Embedding(block_size, n_embd)       # which position it is at
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)          # -> one score per vocabulary token

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)                                    # (B, T, C)
        pos = self.pos_emb(torch.arange(T, device=idx.device))     # (T, C)
        x = tok + pos                                              # (B, T, C)
        x = self.blocks(x)                                         # (B, T, C)
        x = self.ln_f(x)                                           # (B, T, C)
        logits = self.lm_head(x)                                   # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss
print("✅ Ready")

## 4.1 Training: cross-entropy loss, optimizer, batching

### Cross-entropy loss
For each position, the model gives a probability $p$ to the **correct** next token. The loss is simply:

$$\text{loss} = -\ln(p_{\text{correct}})$$

| Probability on the correct token | Loss |
|---|---|
| 0.90 (almost certain and right) | 0.11 |
| 0.50 | 0.69 |
| 1/65 (blind guessing) | 4.17 |
| 0.01 (confident about something wrong) | 4.61 |

The more confident and correct, the smaller it is. Training = **make this number smaller, on average, over all positions**.

In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.1]])      # scores for 3 possible tokens
target = torch.tensor([0])                    # the correct one is token 0

probs = F.softmax(logits, dim=-1)
print("probabilities:", probs.round(decimals=3))
print("-ln(p correct):", round(-torch.log(probs[0, 0]).item(), 4))
print("F.cross_entropy:", round(F.cross_entropy(logits, target).item(), 4))

### Batching
We don't train on the whole text at once, nor on one chunk at a time. At each step we take **`batch_size` random chunks** of length `block_size`:
- the GPU processes them **in parallel**,
- each step "sees" $B \times T$ predictions (e.g. 32 × 32 = 1,024),
- the randomness helps the model not "get used to" a particular order.

In [ ]:
def get_batch(split, batch_size, block_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))          # random starting points
    x = torch.stack([d[i:i + block_size] for i in ix])              # (B, T)
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])      # (B, T), shifted by 1
    return x.to(device), y.to(device)

xb, yb = get_batch("train", batch_size=4, block_size=8)
print("x:", xb.shape)
print(xb)
print("y:", yb.shape)
print(yb)

### Hyperparameters
Two configurations: a small one that finishes quickly on a **CPU**, and a larger one if we have a **GPU**. The cell picks automatically.

In [ ]:
if device == "cuda":
    batch_size, block_size = 64, 128
    n_embd, n_head, n_layer, dropout = 192, 6, 6, 0.1
    max_iters, learning_rate = 3000, 1e-3
else:
    batch_size, block_size = 32, 32
    n_embd, n_head, n_layer, dropout = 64, 4, 4, 0.0
    max_iters, learning_rate = 1500, 1e-3

eval_interval, eval_iters = 250, 50
print(f"device={device} | block_size={block_size} | n_embd={n_embd} | n_layer={n_layer} | iters={max_iters}")

### Measuring the loss on train and validation
We measure the average loss over a few batches. `@torch.no_grad()` tells PyTorch not to keep gradients (here we only measure, we don't train), and `model.eval()` turns off dropout.

- **Train loss falls, val loss falls:** it is learning.
- **Train loss falls, val loss rises:** **overfitting**, i.e. it memorises instead of generalising.

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size, block_size)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

### The training loop
**It is the same 4-step loop from Session 1.** As the optimizer we use **AdamW**, the most common choice for Transformers: it adapts the step size for each parameter on its own.

⏱️ Training takes a little while; on a GPU it is much faster. While it runs, we watch the loss fall from ~4.2.

In [ ]:
torch.manual_seed(1337)
model = MiniGPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

t0 = time.time()
for it in range(max_iters + 1):
    if it % eval_interval == 0:
        l = estimate_loss(model)
        print(f"step {it:5d} | train {l['train']:.3f} | val {l['val']:.3f} | {time.time() - t0:5.0f}s")

    xb, yb = get_batch("train", batch_size, block_size)
    _, loss = model(xb, yb)              # 1. forward + 2. loss
    optimizer.zero_grad(set_to_none=True)
    loss.backward()                      # 3. backward
    optimizer.step()                     # 4. step

## 4.2 Text generation: greedy, temperature, top-k, top-p

The model outputs **logits**. How we choose the next token from them is called the **decoding strategy**, and these are exactly the parameters you will see in every LLM API.

| Strategy | What it does | When |
|---|---|---|
| **Greedy** | Always takes the most likely | Deterministic, but repetitive |
| **Temperature** | Divides the logits by $T$ before softmax. $T<1$: more "confident", $T>1$: more "creative" | The main "creativity knob" |
| **Top-k** | Keeps only the $k$ most likely tokens | Cuts off the crazy choices |
| **Top-p (nucleus)** | Keeps the fewest tokens whose combined probability is $\geq p$ | Adapts: few when confident, many when not |

First, let's see temperature on a small distribution:

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.5, 0.1])
for temp in [0.2, 0.5, 1.0, 2.0]:
    print(f"T={temp}: {F.softmax(logits / temp, dim=-1).round(decimals=2).tolist()}")

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None, greedy=False):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]          # context window
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]                      # last position only: (B, vocab)

        if greedy:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            if top_p is not None:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                cumulative = torch.cumsum(sorted_probs, dim=-1)
                sorted_probs[cumulative - sorted_probs > top_p] = 0.0     # always keep the first
                probs = torch.zeros_like(probs).scatter(-1, sorted_idx, sorted_probs)
                probs = probs / probs.sum(dim=-1, keepdim=True)
            next_id = torch.multinomial(probs, num_samples=1)

        idx = torch.cat([idx, next_id], dim=1)
    return idx

def complete(prompt, max_new_tokens=200, **kwargs):
    idx = torch.tensor([encode(prompt)], device=device)
    return decode(generate(model, idx, max_new_tokens, **kwargs)[0].tolist())

> ⚠️ Our vocabulary contains **only the characters in Shakespeare** (English, no Greek). A prompt with Greek characters will raise a `KeyError`.

In [ ]:
print("=== Greedy ===")
print(complete("ROMEO:", greedy=True))

In [ ]:
print("=== Temperature 0.5 ===")
print(complete("ROMEO:", temperature=0.5))
print("\n=== Temperature 1.5 ===")
print(complete("ROMEO:", temperature=1.5))

In [ ]:
print("=== Top-k = 5 ===")
print(complete("JULIET:", temperature=1.0, top_k=5))
print("\n=== Top-p = 0.9 ===")
print(complete("JULIET:", temperature=1.0, top_p=0.9))

### Try it yourself
- Change the prompt (e.g. `"KING HENRY:"`, `"First Citizen:"`).
- Raise the temperature to 3: what happens? Lower it to 0.1: what does it look like?
- Compare `top_k=1` with greedy. Why do they give the same result?

(Optional) Save the model to load it again later:

In [ ]:
torch.save(model.state_dict(), "minigpt.pt")
print("Saved:", os.path.getsize("minigpt.pt") // 1024, "KB")

## 4.3 Wrap-up: from MiniGPT to today's models

### What is the same and what differs
**The architecture is essentially the same.** What changes is the scale and the steps after pre-training.

| | MiniGPT (us) | GPT-2 small (2019) | Modern large models |
|---|---|---|---|
| Parameters | ~0.2–3 million | 124 million | from a few to hundreds of billions |
| Tokenizer | characters, 65 tokens | BPE, ~50k | BPE/subword, ~100k–250k |
| Context window | 32–128 tokens | 1,024 | tens of thousands to millions of tokens |
| Data | ~1 MB of text | ~40 GB | trillions of tokens |
| Training hardware | a laptop or a free GPU | a few GPUs | thousands of GPUs |

### From base model to assistant
1. **Pre-training:** exactly what we did: next-token prediction on huge amounts of text. The result (*base model*) continues text; it doesn't "chat".
2. **Instruction tuning (SFT):** training on question–answer examples, so that it follows instructions.
3. **Preference tuning (RLHF or similar methods):** it learns from human or automated preferences to be helpful, accurate and safe.

A "conversation" with system/user/assistant messages is ultimately **one long text with special tokens** that the model continues, exactly like our `complete()`.

### The landscape today
- **Closed models via API:** families such as GPT (OpenAI), Claude (Anthropic), Gemini (Google). We pay per input/output token.
- **Open-weight models:** families such as Llama (Meta), Mistral, Qwen, DeepSeek, Gemma. We download the weights and run them wherever we want.
- **Trends:** larger context windows, multimodality (image, audio), models that "think" before answering (reasoning), Mixture-of-Experts, small models that run locally, and agents that use tools.

*(Specific versions change every few months; check the current ones before the class.)*

### From MiniGPT to real applications
| What we built | What you will see in LLM applications and APIs |
|---|---|
| `encode()` / tokens | Per-token pricing, `max_tokens`, context limits |
| `block_size` | Context window: why large documents get "cut off" and we need RAG |
| `temperature`, `top_k`, `top_p` | Exactly the same parameters in the APIs, e.g. `temperature=0` for precision |
| `generate()` loop, token by token | **Streaming**: answers arrive token by token |
| Next-token prediction | Why LLMs can "make things up" (hallucinations): they predict the *likely*, not the *true* |
| `nn.Embedding` | **Embedding models** and vector stores for semantic search |

**Congratulations!** You have built, trained and run a decoder-only Transformer from scratch. From now on, every time you call an LLM, you know what is happening underneath.